使用云端的Qwen-VL对本地的pdf(任意pdf的第一页)进行解析，写一下这个代码;

In [ ]:
import os
import base64
import fitz  # PyMuPDF
from io import BytesIO
from openai import OpenAI


PDF_FILE_PATH = "example.pdf"  # 确保文件路径正确

client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

def get_pdf_page_as_base64(path, page_num=0):
    """
    将 PDF 指定页转换为 Base64 字符串
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"未找到文件: {path}")
    
    doc = fitz.open(path)
    if page_num >= doc.page_count:
        doc.close()
        raise IndexError("页码超出范围")
        
    page = doc.load_page(page_num)
    # 推荐使用 2.0 倍缩放 (约 144 DPI)，保证模型能看清表格细线和微小文字
    pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
    img_data = pix.tobytes("png")  # 使用 PNG 保留更多细节
    doc.close()
    
    base64_str = base64.b64encode(img_data).decode("utf-8")
    return f"data:image/png;base64,{base64_str}"

def analyze_pdf_to_markdown(file_path):
    try:
        # 1. 转换图片
        image_data = get_pdf_page_as_base64(file_path)

        # 2. 调用模型进行结构化解析
        # 系统级提示词触发 Qwen-VL 的结构化解析能力
        prompt = "qwenvl markdown" # 解析为markdown格式，如果是qwenvl html则是解析为html格式
        
        print(f"正在发送请求，使用模型解析为 Markdown 格式...")
        
        completion = client.chat.completions.create(
            model="qwen-vl-max",  # 或者使用 qwen-vl-plus-latest 等支持 Qwen3 系列能力的模型
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": image_data}},
                        {"type": "text", "text": prompt},
                    ],
                },
            ],
            stream=True,
        )

        # 3. 流式接收结果
        print("\n" + "=" * 20 + " 解析结果 " + "=" * 20 + "\n")
        full_result = ""
        for chunk in completion:
            if chunk.choices and chunk.choices[0].delta.content:
                content = chunk.choices[0].delta.content
                print(content, end='', flush=True)
                full_result += content
        
        

        # 4. 写入文件
        if full_result:
            # 自动生成文件名：例如 example.pdf -> example_parsed.md
            base_name = os.path.splitext(file_path)[0]
            output_file = f"{base_name}_parsed.md"
            
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(full_result)
            
            print("\n\n" + "=" * 45)
            print(f"✅ 解析完成！内容已保存至: {output_file}")
            print("=" * 45)

    except Exception as e:
        print(f"\n解析失败: {e}")

if __name__ == "__main__":
    analyze_pdf_to_markdown(PDF_FILE_PATH)

正在发送请求，使用模型解析为 Markdown 格式...

==================== 解析结果 ====================

```markdown
# 南阳理工学院

南理工院文（2026）19号

---

## 关于表彰第六届“校园百佳之星”的决定

校内各单位：

为深入学习贯彻习近平新时代中国特色社会主义思想和党的二十大精神，培育和践行社会主义核心价值观，发挥先进典型的示范引领作用，充分实施“榜样引领计划”，激励学生勤奋学习、努力进取，营造弘扬正气、歌颂真情、倡导善良、崇尚高雅、争当先锋的校园文化氛围，学校于2025年12月启动了第六届“校园百佳之星”评选活动。经学生个人申请，教学院初评、推荐，学校评定、公示，最终评选出我校第六届“校园百佳之星”学生

- 1 -
```

✅ 解析完成！内容已保存至: example_parsed.md
